# 5.4 — Polynomial Regression
### Handling Curved Relationships with Feature Engineering

---

## The Problem Linear Regression Can't Solve

Linear Regression draws a straight line through your data:

$$\hat{y} = w_1x + b$$

This works perfectly when the relationship between feature and output is proportional — double the input, double the output.

But what about **Suresh's used car dealership in Chennai?**

He noticed:
- Cars with very low mileage (nearly new) → very expensive
- As mileage increases → price drops **fast** at first
- After a certain point → price drops **slowly** and plateaus

If you plot mileage vs price — it's a **curve**, not a straight line.

Linear Regression would draw a straight line through this curve and get it systematically wrong — overestimating some prices, underestimating others. No matter how much data you give it, a straight line cannot capture a curve.

**Polynomial Regression fixes this.**

---

## The Core Idea — Feature Engineering

Instead of:
$$\hat{y} = w_1x + b \quad \text{(straight line)}$$

We add higher degree terms:
$$\hat{y} = w_1x + w_2x^2 + w_3x^3 + \cdots + b \quad \text{(curve)}$$

The trick: **x² and x³ are just new features.** We create them from x, then hand everything to plain Linear Regression.

| Original features | After degree=2 | After degree=3 |
|---|---|---|
| mileage | mileage, mileage² | mileage, mileage², mileage³ |

Linear Regression sees `[mileage, mileage², mileage³]` as three separate features — it has no idea they came from the same original feature. It just finds the best weights w₁, w₂, w₃ as usual.

**The curve comes from the data transformation — not from a new algorithm.**

---

## What "Linear" Really Means

The word **linear** in Linear Regression means: **linear in the coefficients** — not linear in the features.

- w₁, w₂, w₃ are still just multiplied and added. No w² or w³.
- That's why plain Linear Regression can learn polynomial relationships — as long as you engineer the features first.

---

## Interaction Terms

When you have multiple features and apply PolynomialFeatures, it also creates **interaction terms**.

For features `[mileage, age]` with degree=2:
```
[1, mileage, age, mileage², mileage×age, age²]
```

The `mileage × age` term captures: *"how do mileage and age combined affect price?"*

A 10-year-old car with 200,000 km is worth less than a 10-year-old car with 50,000 km — the combination matters, not just each feature alone.

---

## The Feature Explosion Problem

More degree = more combinations = more features:

| Original features | Degree | Total features after PolynomialFeatures |
|---|---|---|
| 1 | 2 | 3 |
| 1 | 3 | 4 |
| 2 | 2 | 6 |
| 2 | 3 | 10 |
| 5 | 2 | 21 |
| 5 | 3 | 56 |

More features = more complex model = **overfitting risk**.

This is why **degree is the key hyperparameter** — just like alpha in Ridge/Lasso.

| Degree | Effect |
|--------|--------|
| 1 | Straight line — underfits curved data |
| 2-3 | Gentle curve — usually just right |
| 5+ | Starts overfitting |
| 10+ | Wildly twisting curve — memorises noise |

We find the best degree using **cross-validation** — same logic as alpha in Ridge/Lasso.

---

## StandardScaler — Why It's Mandatory Here

StandardScaler transforms every feature to mean=0, std=1:

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

Without scaling, polynomial terms explode in size:
- mileage = 150,000
- mileage² = 22,500,000,000
- mileage³ = 3,375,000,000,000,000

Gradient descent completely breaks down with numbers this large — the steps become wildly uneven. Scaling keeps all features in a reasonable range.

---

## Pipeline — The Right Order

Pipeline chains steps so output of one becomes input of next — and **prevents data leakage** by ensuring fitting only happens on training data.

```python
Pipeline([
    ('poly',   PolynomialFeatures(degree=2)),  # Step 1: create x², interactions
    ('scaler', StandardScaler()),               # Step 2: scale all expanded features
    ('model',  LinearRegression())              # Step 3: learn weights
])
```

**Why Poly before Scaler?**
PolynomialFeatures needs raw original features to create x², x³. It decides how many features exist. Then StandardScaler scales ALL of those expanded features fairly.

If Scaler came first → it scales original features → Poly creates polynomial terms from already-scaled values → inconsistent scaling.

**Correct order always: PolynomialFeatures → StandardScaler → Model**

---

## Real World Problem — Chennai Used Car Price Prediction

**Suresh** runs a used car dealership in Chennai. He wants a model that predicts the resale price (₹ lakhs) of a car based on:
- `mileage_km` — total kilometres driven
- `age_years` — how old the car is
- `engine_cc` — engine size
- `service_records` — number of service records available

The mileage-price relationship is curved — not linear. Polynomial Regression is the right choice.

**We will:**
1. Show why Linear Regression fails on curved data
2. Show how Polynomial Regression fixes it
3. Compare different degrees — underfitting vs overfitting
4. Find best degree using cross-validation
5. Build the full pipeline and evaluate

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# WHY these imports:
# PolynomialFeatures — creates x², x³, interaction terms automatically
# Pipeline          — chains poly + scaler + model, prevents data leakage
# cross_val_score   — find best degree using CV (never use test set for this)
# LinearRegression  — the actual learning algorithm (Poly just engineers features)

np.random.seed(42)

In [ ]:
# ── Step 1: Create Dataset — Curved Relationship ──────────────────────────────
n = 600  # 600 used cars

mileage_km      = np.random.randint(5000, 200000, n)
age_years       = np.random.randint(1, 20, n)
engine_cc       = np.random.choice([800, 1000, 1200, 1500, 1800, 2000], n)
service_records = np.random.randint(0, 15, n)

# TRUE price formula — CURVED relationship with mileage
# Price drops fast at first (high mileage effect) then plateaus
# We model this using log(mileage) — but we'll pretend we don't know this
# and let Polynomial Regression discover the curve
noise = np.random.normal(0, 1.5, n)

price_lakhs = (
    25                                    # base price
    - 0.00008  * mileage_km               # linear mileage effect
    - 0.000000000003 * mileage_km**2      # curved mileage effect (quadratic)
    - 0.8      * age_years                # older = cheaper
    + 0.003    * engine_cc                # bigger engine = more expensive
    + 0.2      * service_records          # more records = trustworthy = more expensive
    + noise
).clip(1, 40)  # keep prices realistic

df = pd.DataFrame({
    'mileage_km':      mileage_km,
    'age_years':       age_years,
    'engine_cc':       engine_cc,
    'service_records': service_records,
    'price_lakhs':     np.round(price_lakhs, 2)
})

print(f"Dataset shape: {df.shape}")
print(f"\nPrice range: ₹{df['price_lakhs'].min():.2f}L to ₹{df['price_lakhs'].max():.2f}L")
print(f"Average price: ₹{df['price_lakhs'].mean():.2f}L")
df.head()

In [ ]:
# ── Step 2: Visualise the Curved Relationship ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mileage vs Price — the curved relationship
axes[0].scatter(df['mileage_km'], df['price_lakhs'],
                alpha=0.3, color='steelblue', s=15)
axes[0].set_xlabel('Mileage (km)')
axes[0].set_ylabel('Price (₹ lakhs)')
axes[0].set_title('Mileage vs Price — Clearly Curved\n(Linear Regression will fail here)')

# Age vs Price — more linear
axes[1].scatter(df['age_years'], df['price_lakhs'],
                alpha=0.3, color='tomato', s=15)
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Price (₹ lakhs)')
axes[1].set_title('Age vs Price — More Linear\n(Linear Regression handles this fine)')

plt.tight_layout()
plt.show()

# WHY visualise first?
# Always look at your data before choosing an algorithm.
# The mileage plot shows a clear curve — this is the visual justification for
# choosing Polynomial Regression over plain Linear Regression.

In [ ]:
# ── Step 3: Split Data ────────────────────────────────────────────────────────
X = df.drop('price_lakhs', axis=1)
y = df['price_lakhs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} cars | Test: {X_test.shape[0]} cars")

In [ ]:
# ── Step 4: What PolynomialFeatures Actually Creates ─────────────────────────
# Before using it in a pipeline, let's see what it does under the hood

poly_demo = PolynomialFeatures(degree=2, include_bias=False)
# WHY include_bias=False?
# By default PolynomialFeatures adds a column of 1s (the bias/intercept term)
# LinearRegression already handles the intercept internally — so we skip it here

# Show on a tiny example — 2 features, 3 rows
demo_data = np.array([[100, 2],
                       [200, 5],
                       [150, 3]])
demo_transformed = poly_demo.fit_transform(demo_data)

print("Original features: [mileage, age]")
print(demo_data)
print(f"\nAfter PolynomialFeatures(degree=2):")
print(f"Feature names: {poly_demo.get_feature_names_out(['mileage', 'age'])}")
print(demo_transformed)
print(f"\nOriginal: 2 features → After Poly(degree=2): {demo_transformed.shape[1]} features")
print("\nNew features created:")
print("  mileage²      — captures curved mileage effect")
print("  mileage×age   — interaction: combined effect of mileage AND age")
print("  age²          — captures curved age effect")

In [ ]:
# ── Step 5: Degree Comparison — Underfitting vs Overfitting ───────────────────
# Use only mileage for this demo (1 feature = easy to visualise)

X_mile_train = X_train[['mileage_km']]
X_mile_test  = X_test[['mileage_km']]

degrees     = [1, 2, 3, 6, 10]
train_rmses = []
test_rmses  = []

fig, axes = plt.subplots(1, len(degrees), figsize=(18, 4))
x_plot = np.linspace(X_mile_train['mileage_km'].min(),
                     X_mile_train['mileage_km'].max(), 300).reshape(-1, 1)
x_plot_df = pd.DataFrame(x_plot, columns=['mileage_km'])

for i, degree in enumerate(degrees):
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ])
    pipe.fit(X_mile_train, y_train)

    train_pred = pipe.predict(X_mile_train)
    test_pred  = pipe.predict(X_mile_test)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse  = np.sqrt(mean_squared_error(y_test,  test_pred))
    train_rmses.append(train_rmse)
    test_rmses.append(test_rmse)

    # Plot the fitted curve
    y_plot = pipe.predict(x_plot_df)
    axes[i].scatter(X_mile_train['mileage_km'], y_train,
                    alpha=0.2, s=10, color='steelblue')
    axes[i].plot(x_plot, y_plot, color='red', linewidth=2)
    axes[i].set_title(f'Degree {degree}\nTrain RMSE: {train_rmse:.2f}\nTest RMSE: {test_rmse:.2f}')
    axes[i].set_xlabel('Mileage (km)')
    axes[i].set_ylabel('Price (₹L)')
    axes[i].set_ylim(0, 40)

plt.suptitle('Effect of Polynomial Degree — Underfitting to Overfitting', y=1.02)
plt.tight_layout()
plt.show()

# WHY this plot?
# Degree 1: straight line through curved data — clearly underfitting
# Degree 2-3: gentle curve — fits the data well
# Degree 6+: wiggly curve chasing individual points — overfitting
# The best degree has LOW test RMSE — not just low train RMSE

In [ ]:
# ── Step 6: Train RMSE vs Test RMSE — The Overfitting Signature ───────────────
plt.figure(figsize=(8, 4))
plt.plot(degrees, train_rmses, 'o-', color='steelblue', label='Train RMSE', linewidth=2)
plt.plot(degrees, test_rmses,  'o-', color='tomato',    label='Test RMSE',  linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE (₹ lakhs)')
plt.title('Train vs Test RMSE — Finding the Sweet Spot')
plt.legend()
plt.xticks(degrees)
plt.tight_layout()
plt.show()

# WHY this plot?
# This is the classic overfitting signature:
# - Train RMSE keeps falling as degree increases (model memorises more)
# - Test RMSE falls at first, then RISES (model memorises noise)
# - The best degree = lowest point on the TEST RMSE curve
# - Gap between train and test RMSE = how much the model is overfitting
print("\nDegree | Train RMSE | Test RMSE")
print("-" * 35)
for d, tr, te in zip(degrees, train_rmses, test_rmses):
    print(f"  {d:<6} | ₹{tr:>6.3f}L   | ₹{te:>6.3f}L")

In [ ]:
# ── Step 7: Find Best Degree with Cross-Validation ────────────────────────────
# We use ALL features now (not just mileage)
# CV finds the best degree without touching the test set

degrees_to_try = [1, 2, 3, 4, 5]
cv_scores = []

for degree in degrees_to_try:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model',  LinearRegression())
    ])

    # cross_val_score runs 5-fold CV on training data
    # scoring='neg_root_mean_squared_error' → returns negative RMSE
    # WHY negative? sklearn convention — higher score = better.
    # Since lower RMSE = better, it negates RMSE so higher = lower error.
    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    cv_rmse = -scores.mean()  # negate back to get positive RMSE
    cv_scores.append(cv_rmse)
    print(f"Degree {degree}: CV RMSE = ₹{cv_rmse:.4f}L (±{scores.std():.4f})")

best_degree = degrees_to_try[np.argmin(cv_scores)]
# WHY np.argmin? — finds the index of the lowest CV RMSE
print(f"\nBest degree: {best_degree} (lowest CV RMSE)")

In [ ]:
# ── Step 8: Train Final Model with Best Degree ────────────────────────────────
final_pipeline = Pipeline([
    ('poly',   PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

final_pipeline.fit(X_train, y_train)
# WHY fit on full X_train (not scaled separately)?
# Pipeline handles everything internally:
#   1. PolynomialFeatures fits+transforms X_train
#   2. StandardScaler fits+transforms the expanded X_train
#   3. LinearRegression trains on the scaled expanded X_train
# No manual steps needed — and no leakage possible.

y_pred = final_pipeline.predict(X_test)
# WHY just predict (not transform manually)?
# Pipeline applies all steps in order automatically:
#   1. PolynomialFeatures transforms X_test (using what it learned from train)
#   2. StandardScaler transforms (using mean/std from train)
#   3. LinearRegression predicts

final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
final_r2   = r2_score(y_test, y_pred)

# How many features after polynomial expansion?
n_features_after = final_pipeline.named_steps['poly'].n_output_features_

print(f"Final Model — Polynomial Degree {best_degree}")
print(f"Original features : {X_train.shape[1]}")
print(f"After Poly expand : {n_features_after} features")
print(f"\nTest RMSE : ₹{final_rmse:.2f} lakhs")
print(f"Test R²   : {final_r2:.4f}")
print(f"\nOn average, predictions are off by ₹{final_rmse:.2f} lakhs")

In [ ]:
# ── Step 9: Compare Linear vs Polynomial ─────────────────────────────────────
# Linear Regression baseline
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
lr_pipe.fit(X_train, y_train)
lr_pred  = lr_pipe.predict(X_test)
lr_rmse  = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2    = r2_score(y_test, lr_pred)

print("=" * 52)
print("MODEL COMPARISON — CHENNAI USED CAR PRICES")
print("=" * 52)
print(f"{'Model':<30} {'RMSE':>8} {'R²':>8}")
print("-" * 52)
print(f"{'Linear Regression':<30} ₹{lr_rmse:>5.2f}L  {lr_r2:>6.4f}")
print(f"{'Polynomial (degree='+str(best_degree)+')':<30} ₹{final_rmse:>5.2f}L  {final_r2:>6.4f}")
print("=" * 52)
improvement = ((lr_rmse - final_rmse) / lr_rmse) * 100
print(f"\nPolynomial Regression improved RMSE by {improvement:.1f}%")
print("This improvement comes entirely from capturing the curved mileage relationship.")

In [ ]:
# ── Step 10: Visualise Final Predictions ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=20)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (₹ lakhs)')
axes[0].set_ylabel('Predicted Price (₹ lakhs)')
axes[0].set_title(f'Polynomial (degree={best_degree}) — Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, color='tomato', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Price (₹ lakhs)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot — Random scatter = good fit')

plt.tight_layout()
plt.show()

In [ ]:
# ── Step 11: Sample Predictions — Suresh's Use Case ──────────────────────────
# Suresh wants to price these specific cars:
sample_cars = pd.DataFrame({
    'mileage_km':      [15000,  80000,  150000, 190000],
    'age_years':       [1,      5,      10,     15],
    'engine_cc':       [1200,   1500,   1000,   800],
    'service_records': [2,      8,      5,      3]
})

sample_preds = final_pipeline.predict(sample_cars)

print("Suresh's Price Estimates:")
print("-" * 65)
print(f"{'Mileage':>10} {'Age':>5} {'Engine':>8} {'Services':>10} {'Est. Price':>12}")
print("-" * 65)
for i, (_, row) in enumerate(sample_cars.iterrows()):
    print(f"{int(row['mileage_km']):>10,} {int(row['age_years']):>5}yr "
          f"{int(row['engine_cc']):>6}cc {int(row['service_records']):>9} "
          f"   ₹{sample_preds[i]:>6.2f} lakhs")

print("\nNotice: high mileage + old age = low price, as expected.")
print("The curved mileage effect is captured — price drops fast initially, then slows.")

---

## Summary Table

| | Polynomial Regression |
|---|---|
| **Task** | Regression with curved/non-linear relationships |
| **How it works** | Engineers x², x³, interaction terms → feeds to Linear Regression |
| **Key hyperparameter** | `degree` — controls how complex the curve can be |
| **Key class** | `PolynomialFeatures` — creates new features automatically |
| **Pipeline order** | PolynomialFeatures → StandardScaler → LinearRegression |
| **Why scale?** | x², x³ produce huge numbers — gradient descent breaks without scaling |
| **How to pick degree** | Cross-validation on training data — never use test set |
| **Degree too low** | Underfits — misses the curve |
| **Degree too high** | Overfits — chases noise, wiggly curve |
| **Strength** | Captures non-linear relationships with no new algorithm |
| **Weakness** | Feature explosion — too many features with high degree + many inputs |
| **When to use** | Visual inspection shows curved relationship between feature and output |

---

## What's Next?

So far all regression algorithms have been variations of the linear framework.

**5.5 Decision Tree Regression** is completely different — it doesn't use equations or coefficients at all. Instead it splits data into rectangular regions and predicts the **mean value** of training points in each region. Same splitting logic as Chapter 4's Decision Tree classifier — but instead of a class label, each leaf predicts a number.

---

## Practice Task

Priya works at an agricultural research centre in Coimbatore. She's studying how fertiliser amount affects crop yield (kg per acre). The relationship is curved — too little fertiliser = low yield, optimal amount = peak yield, too much = yield drops (toxicity).

Features:
- `fertiliser_kg` — fertiliser applied per acre
- `rainfall_mm` — rainfall received
- `soil_quality` — soil quality score (1-10)
- `temperature_c` — average temperature

**Your tasks:**

1. Create synthetic dataset of 500 farms with a curved fertiliser-yield relationship
2. Plot fertiliser vs yield to confirm the curve
3. Compare Linear Regression vs Polynomial (degree 2, 3, 5) — plot all curves
4. Find best degree using cross-validation
5. Train final pipeline and evaluate on test set
6. What degree did CV choose? Does it make sense visually?

In [ ]:
# YOUR CODE HERE

# Step 1: Create dataset with curved fertiliser-yield relationship

# Step 2: Plot fertiliser vs yield

# Step 3: Compare Linear vs Polynomial degrees

# Step 4: Cross-validation to find best degree

# Step 5: Train final pipeline and evaluate

# Step 6: Answer — what degree did CV choose?